In [8]:

# -*- coding: utf-8 -*-
"""
Balanced Oil and Vinegar Şemasına Yönelik Kipnis-Shamir Saldırısı
==================================================================

Bu modül, Patarin'in 1997 yılında önerdiği Balanced Oil and Vinegar (v = o)
imza şemasına karşı Kipnis ve Shamir'in 1998 yılında yayımladıkları
"Cryptanalysis of the Oil and Vinegar Signature Scheme" başlıklı
çalışmada tanımlanan polinom-zamanlı yapısal saldırının (kısaca
Kipnis-Shamir saldırısı, KS saldırısı) öğretici bir SageMath
simülasyonunu içermektedir.

Teorik Arka Plan
-----------------
Bir önceki bölümde ele alınan OV/UOV şemasında açık anahtar, gizli
merkezi dönüşüm F ile gizli tersinir afin dönüşüm T'nin bileşkesi
olarak P = F o T biçiminde tanımlanır. Merkezi dönüşümün her bir
bileşeni, kuadratik form matrisi cinsinden

        [ V x V bloğu   |  V x O bloğu ]
    M = [---------------+--------------]
        [ (simetriği)   |   0 (o x o)  ]

yapısındadır; yani Oil x Oil terimleri sıfırdır. T dönüşümü uygulandığında
açık anahtar matrisleri P_k = T^T * M_k * T haline gelir ve bu sıfır blok
yapısı GÖRÜNÜRDE tamamen kaybolur; P_k matrisleri saldırgana rastgele
görünür.

Kipnis-Shamir'in temel gözlemi, bu sıfır blok yapısının DOĞRUSAL CEBİR
düzeyinde tamamen yok olmadığı, yalnızca GİZLENDİĞİdir. Balanced OV
durumunda (v = o, dolayısıyla n = 2o), açık anahtar matrislerinden uygun
iki tanesinin (W1, W2) doğrusal kombinasyonu alınıp W1 tersinir olacak
şekilde seçildiğinde, W = W1^{-1} * W2 matrisinin gizli Oil uzayını bir
DEĞİŞMEZ ALT UZAY (invariant subspace) olarak barındırdığı gösterilebilir.
Bu, klasik "matris demeti" (matrix pencil) tekniğinin çok değişkenli
kriptografiye uyarlanmış halidir.

Daha somut olarak, W matrisinin karakteristik polinomu C(lambda), q tek
karakteristikli balanced OV durumunda TAM KARE bir yapı sergiler:

    C(lambda) = C1(lambda)^2

Bu cebirsel gerçek, merkezi dönüşümün blok yapısından kaynaklanan bir
simetriden ileri gelir. C1(lambda) = sqrt(C(lambda)) polinomu hesaplanıp
C1(W) matrisi elde edildiğinde, bu matrisin sağ çekirdeği (right kernel),
gizli Oil uzayına ait vektörleri (veya bu uzayla örtüşen bir aday alt
uzayı) içerir. Böylece saldırgan, yalnızca AÇIK anahtara erişerek,
üstel karmaşıklıkta arama yapmadan, POLİNOM ZAMANDA gizli Oil uzayının
bir temsilini yeniden inşa edebilir.

Gizli Oil uzayı bir kez bulunduğunda, saldırgan bu uzayı kullanarak
orijinal (T, F) anahtar çiftine matematiksel olarak EŞDEĞER bir
(T', F') anahtar çifti inşa edebilir (equivalent keys). Bu denk anahtar
çifti ile, orijinal gizli anahtara hiç erişmeden, herhangi bir mesaj
için geçerli bir sahte imza (forgery) üretilebilir. İşte bu son adım,
saldırının pratik kriptografik anlamda "sistemi kırma" niteliğini
tamamlar.

Bu saldırı tarihsel olarak son derece önemlidir: Kipnis-Shamir
saldırısının balanced OV şemasını polinom zamanda kırması, literatürün
neden UNBALANCED OV (v > o, kısaca UOV) yapısına yöneldiğinin doğrudan
gerekçesidir. Vinegar değişken sayısı Oil değişken sayısından yeterince
büyük seçildiğinde (v > o), W1^{-1} * W2 matrisinin karakteristik
polinomu artık tam kare yapısını korumaz ve bu saldırı doğrudan
uygulanamaz hale gelir.

Bu Simülasyonun Sınırları ve Amaçları
---------------------------------------
- Bu kod GÜVENLİ parametreler üretmek için değil, saldırının ARDINDAKİ
  temel doğrusal cebirsel fikri somut biçimde göstermek amacıyla
  hazırlanmıştır.
- Öğretici sadeliği korumak için v = o (balanced OV) seçilmiştir;
  gerçek dünyada kullanılan UOV varyantları v > o koşuluna dayanır ve
  bu saldırıya karşı (bu haliyle) dayanıklıdır.
- Merkezi dönüşüm polinomları, saldırının matris-demeti fikrini net
  biçimde göstermek amacıyla yalnızca HOMOJEN kuadratik terimlerden
  oluşturulmuştur; doğrusal ve sabit terimler eklenmemiştir.
- q'nun TEK (tek asal kuvveti) olduğu varsayılmıştır; karakteristik 2
  durumunda simetrik matris temsili ve çapraz terimlerin (2*x_i*x_j
  yerine karakteristik 2'de x_i*x_j) farklı ele alınması gerektiğinden,
  bu kodun kapsamı dışında tutulmuştur.
- Kodda gizli Oil uzayı (Secret_Oil_Space) yalnızca SİMÜLASYON
  DOĞRULAMASI amacıyla saklanmaktadır; gerçek bir saldırgan bu bilgiye
  asla erişemez ve saldırı yalnızca açık anahtar bilgisiyle yürütülür.

Algoritmanın Genel Akışı
-------------------------
1. Sistem İnşası (generate_keys):
   Gizli T dönüşümü, gizli Oil uzayı (T^{-1} aracılığıyla) ve gizli
   merkezi dönüşüm matrisleri F_k rastgele üretilir; bunlardan açık
   anahtar matrisleri P_k = T^T * F_k * T hesaplanır.

2. Saldırı (run_attack):
   a) Açık anahtar matrislerinin rastgele doğrusal kombinasyonlarından
      W1 (tersinir) ve W2 matrisleri oluşturulur.
   b) W = W1^{-1} * W2 hesaplanır; karakteristik polinomunun tam kare
      olduğu (C(lambda) = C1(lambda)^2) doğrulanır.
   c) C1(W) matrisinin sağ çekirdeği, gizli Oil uzayına dair aday bir
      alt uzay verir.
   d) Aday vektörler, açık anahtar matrisleriyle (u^T P_k w = 0 testi)
      doğrulanarak gerçek Oil uzayı adım adım yeniden inşa edilir.
   e) Bulunan Oil uzayı kullanılarak denk bir (T', F') anahtar çifti
      inşa edilir.
   f) Bu denk anahtar çifti ile rastgele bir mesaj için sahte imza
      üretilir ve orijinal açık anahtarla doğrulanarak saldırının
      başarısı kanıtlanır.

Bu implementasyon; yüksek lisans tezi kapsamında Kipnis-Shamir
saldırısının kavramsal ve doğrusal cebirsel yapısını somutlaştırmak
amacıyla eğitim/araştırma amaçlı geliştirilmiştir.

-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), çok değişkenli polinom halkaları (PolynomialRing), vektör
uzayları (VectorSpace), matris cebiri ve karakteristik polinom
çarpanlarına ayırma gibi SageMath'in yerleşik simgesel/sayısal cebir
araçlarına dayanır.



Referans: Tez, Bölüm 6
"""

from sage.all import *

class KipnisShamirAttackDemo:
    """
    Balanced Oil and Vinegar şemasına yönelik Kipnis-Shamir saldırısının
    öğretici SageMath simülasyonu.

    Bu kod güvenli parametreler üretmek amacıyla değil, saldırının temel
    lineer cebirsel fikrini göstermek amacıyla hazırlanmıştır.

    Not:
    - Bu örnekte v = o alınarak balanced OV yapısı kurulmaktadır.
    - Merkezi polinomlar homojen kuadratik biçimde seçilmiştir.
    - Gerçek saldırgan gizli oil uzayını bilmez; kodda gizli oil uzayı yalnızca
      simülasyon doğrulaması için tutulmaktadır.

    Matematiksel Kurulum
    ---------------------
    - F_q  : Temel sonlu cisim (q TEK, yani karakteristik 2 değil). Bu
      varsayım, kuadratik formların simetrik matrislerle temsil
      edilebilmesi ve çapraz terimlerin 2*x_i*x_j biçiminde tutarlı
      biçimde ifade edilebilmesi için gereklidir.
    - R = F_q[x_0, ..., x_{n-1}] : n = v + o = 2o değişkenli, F_q
      katsayılı çok değişkenli polinom halkası.
    - Değişken bölünmesi: x_0, ..., x_{v-1} Vinegar değişkenleri;
      x_v, ..., x_{n-1} Oil değişkenleridir (balanced OV'de v = o).

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (F_q); bu simülasyonda TEK
        karakteristikli olması beklenir.
    o : int
        Oil değişkenlerinin (ve merkezi dönüşüm bileşenlerinin) sayısı.
        Balanced OV varsayımı gereği Vinegar değişken sayısı da o'ya
        eşit alınır (v = o).

    Öznitelikler (Attributes)
    --------------------------
    v : int
        Vinegar değişkenlerinin sayısı; bu sınıfta her zaman o'ya eşittir
        (v = o, balanced OV).
    n : int
        Toplam değişken sayısı (n = o + v = 2o).
    F : FiniteField
        Temel sonlu cisim F_q.
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    vars : vector
        R halkasının üreteçlerinden (x_0, ..., x_{n-1}) oluşan vektör.
    T_mat : Matrix
        Gizli tersinir n x n afin (bu örnekte yalnızca doğrusal) dönüşüm
        matrisi.
    P_matrices : list of Matrix
        Açık anahtarın her bir P_k bileşenine ait kuadratik form matrisi.
    P_polys : list of MPolynomial
        Açık anahtarı oluşturan o adet homojen kuadratik polinom.
    Secret_Oil_Space : VectorSpace
        Gizli Oil uzayının T_mat aracılığıyla F_q^n içine taşınmış hali;
        YALNIZCA simülasyon doğrulaması amacıyla saklanır, saldırı
        sürecinde KULLANILMAZ.
    """
    def __init__(self, q, o):
        self.q = q
        self.o = o
        # Kipnis-Shamir saldırısı klasik olarak balanced OV durumunu hedefler.
        # Bu nedenle bu öğretici örnekte v = o alınmıştır.
        self.v = o
        self.n = self.o + self.v

        # F_q temel sonlu cismi ve n = 2o değişkenli çok değişkenli
        # polinom halkası; hem gizli merkezi dönüşüm hem de açık anahtar
        # polinomlarının ortak tanım kümesidir.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = vector(self.R, self.R.gens())

        print("\n" + "="*80)
        print(f"KIPNIS-SHAMIR SALDIRISI: DÜZELTİLMİŞ TAM ANALİZ")
        print(f"Parametreler: GF({q}), o={o}, v={self.v} (Balanced), n={self.n}")
        print("="*80)

        # Gizli ve açık anahtar bileşenleri generate_keys() çağrılana
        # kadar boş/None tutulur; Secret_Oil_Space yalnızca simülasyon
        # doğrulaması amaçlı bir "yan kanal" bilgisidir.
        self.T_mat = None
        self.P_matrices = []
        self.P_polys = []
        self.Secret_Oil_Space = None

    def generate_keys(self):
        """
        Balanced OV şemasının gizli anahtarını (T dönüşümü, gizli Oil
        uzayı ve merkezi dönüşüm F) rastgele üretir ve bunlardan açık
        anahtarı (P_matrices, P_polys) türetir.

        Algoritmanın Adımları
        -----------------------
        1. Gizli T Dönüşümü:
           n x n boyutunda TERSİNİR rastgele bir matris T_mat seçilir.
           Bu dönüşüm, merkezi dönüşümün Oil/Vinegar blok yapısını
           saldırgandan gizleyen maskeleme katmanıdır.

        2. Gizli Oil Uzayı:
           Merkezi (gizli) koordinat sisteminde Oil uzayı, doğal olarak
           standart baz vektörlerinin son o tanesi (e_v, ..., e_{n-1})
           tarafından gerilir (bu, merkezi dönüşümün Oil x Oil bloğunun
           sıfır olmasının geometrik karşılığıdır). T dönüşümü açık
           koordinat sistemine T^{-1} ile taşındığında,
                Secret_Oil_Space = span{ T^{-1} * e_v, ..., T^{-1} * e_{n-1} }
           elde edilir. Bu uzay, saldırının NİHAİ HEDEFİDİR; ancak
           yalnızca simülasyon doğrulaması için saklanır, saldırı
           algoritması bu bilgiyi KULLANMAZ.

        3. Merkezi Dönüşüm (F) ve Açık Anahtar (P):
           Her k için, sağ alt o x o (Oil x Oil) bloğu sıfır bırakılan
           SİMETRİK bir matris M_k rastgele üretilir (q tek karakteristikli
           seçildiği için simetrik matris temsili ve çapraz terimlerdeki
           2 katsayısı tutarlı biçimde işler). Bu matristen homojen
           kuadratik polinom poly_f = x^T * M_k * x elde edilir (öğretici
           sadelik için doğrusal/sabit terim eklenmez). Açık anahtar
           matrisi, P_k = T_mat^T * M_k * T_mat eşitliğiyle hesaplanır;
           bu dönüşüm sonucunda M_k'daki sıfır blok yapısı P_k üzerinde
           GÖRÜNMEZ hale gelir (saldırının kırmaya çalıştığı gizleme
           budur).

        Yan Etkiler
        -----------
        self.T_mat, self.Secret_Oil_Space, self.P_matrices ve
        self.P_polys öznitelikleri bu metot tarafından (yeniden)
        doldurulur. Metot birden fazla kez çağrılırsa önceki anahtar
        bileşenleri temizlenip yeniden üretilir. Süreç boyunca ayrıntılı
        ara çıktılar ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        # generate_keys() birden fazla kez çağrılırsa eski veriler temizlenir.
        self.P_matrices = []
        self.P_polys = []
        self.Secret_Oil_Space = None

        print("\n" + "*"*30 + " ADIM 1: SİSTEM İNŞASI (KEYGEN) " + "*"*30)

        # 1. GİZLİ T
        # T_mat, gizli merkezi dönüşümün Oil/Vinegar blok yapısını
        # saldırgandan gizleyen tersinir bir doğrusal dönüşümdür.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break

        print(f"\n[1.1] Gizli T Matrisi (Değişken Karıştırıcı):\n{self.T_mat}")

        # 2. GİZLİ OIL UZAYI
        # Merkezi (gizli) koordinat sisteminde Oil uzayı, standart baz
        # vektörlerinin son o tanesi tarafından gerilir; bu, merkezi
        # dönüşümün Oil x Oil bloğunun sıfır olmasının doğal sonucudur.
        central_oil_basis = []
        for i in range(self.v, self.n):
            vec = vector(self.F, [0]*self.n)
            vec[i] = 1
            central_oil_basis.append(vec)

        # T^{-1} uygulanarak bu uzay, açık (P ile çalışılan) koordinat
        # sistemine taşınır; saldırının nihai hedefi bu uzayı YENİDEN
        # İNŞA ETMEKTİR (ancak yalnızca açık anahtar bilgisiyle).
        T_inv = self.T_mat.inverse()
        secret_basis_vectors = [T_inv * vec for vec in central_oil_basis]
        self.Secret_Oil_Space = VectorSpace(self.F, self.n).span(secret_basis_vectors)

        print(f"\n[1.2] Gizli Oil Uzayı O (Hedef):")
        print(f"   Baz Vektörleri:\n{self.Secret_Oil_Space.basis_matrix()}")

        # 3. MERKEZİ DÖNÜŞÜM (F) ve AÇIK ANAHTAR (P)
        print(f"\n[1.3] Gizli (F) ve Açık (P) Bileşenlerin İnşası")

        for k in range(self.o):
            print(f"\n   >>> Bileşen k={k} <<<")

            # A) Gizli Matris F_k
            # Sağ alt oil-oil bloğu sıfır bırakılır.
            # Matris simetrik seçilmiştir; böylece kuadratik form x^T M_k x olarak okunabilir.
            # q tek karakteristikli seçildiği için çapraz terimlerde 2 katsayısı oluşabilir.
            M_k = matrix(self.F, self.n, self.n)
            for i in range(self.v):
                for j in range(i, self.n):
                    val = self.F.random_element()
                    M_k[i, j] = val
                    M_k[j, i] = val
            # Bu simülasyonda saldırının matris-demeti fikrini göstermek için
            # yalnızca homojen kuadratik kısım kullanılmıştır.
            # Lineer ve sabit terimler eklenmemiştir.
            poly_f = self.vars * M_k * self.vars
            print(f"   Gizli F_{k} Matrisi (Sağ Alt Blok Sıfır):\n{M_k}")

            # B) Açık Matris P_k
            # T dönüşümü uygulandığında sıfır blok yapısı görünürde
            # tamamen kaybolur; ancak bu yapı doğrusal cebir düzeyinde
            # (W1^{-1} * W2'nin özyapısında) hâlâ mevcuttur -- saldırının
            # sömürdüğü nokta tam olarak burasıdır.
            P_k = self.T_mat.transpose() * M_k * self.T_mat
            self.P_matrices.append(P_k)

            # C) Açık Polinom
            poly_p = self.vars * P_k * self.vars
            self.P_polys.append(poly_p)

            print(f"   Açık P_{k} Matrisi (Sıfırlar Kayboldu!):\n{P_k}")
            
            # --- AÇIK ANAHTAR POLİNOMUNUN YAZDIRILMASI ---
            # P_k matrisinden elde edilen açık anahtar polinomu, saldırganın
            # gördüğü TEK bilgidir; formülü P_k(x) = x^T * P_k * x biçimindedir
            # ve içindeki Oil x Oil terimlerinin (gizli F_k'nin aksine) SIFIR
            # OLMADIĞI, dolayısıyla polinomun görünürde tamamen rastgele bir
            # kuadratik form gibi göründüğü bu çıktıda gözlemlenebilir.
            print(f"   Açık Anahtar Polinomu Formülü: P_{k}(x) = x^T * P_{k} * x")
            print(f"   Açık Anahtar Polinomu P_{k}(x) (Açık Hali):")
            print(f"      {poly_p}")

    def _is_public_oil_candidate(self, space):
        """
        Bir aday alt uzayın oil uzayı olup olamayacağını açık anahtar matrisleriyle test eder.

        Kontrol:
        Her P_k için ve aday uzayın baz vektörleri u,w için
        u^T P_k w = 0 olmalıdır.

        Bu kontrol saldırganın erişebildiği açık anahtar bilgisiyle yapılır.

        Teorik Gerekçe
        ---------------
        Gizli Oil uzayının tanımlayıcı özelliği, merkezi koordinatlarda
        Oil x Oil bloğunun sıfır olmasıdır; yani u, w bu uzaydan
        alındığında u^T * M_k * w = 0 her zaman geçerlidir. P_k = T^T *
        M_k * T dönüşümü bir kongrüans (congruence) dönüşümü olduğundan,
        bu sıfırlama özelliği KOORDİNAT DEĞİŞİMİNDEN BAĞIMSIZ olarak
        korunur: u', w' açık koordinatlardaki Oil uzayı vektörleri ise
        u'^T * P_k * w' = 0 da geçerlidir. Bu değişmezlik, saldırganın
        yalnızca açık P_k matrislerini kullanarak bir aday uzayın gerçek
        Oil uzayı olup olmadığını test edebilmesinin temelidir.

        Parametreler
        ------------
        space : VectorSpace
            Test edilecek aday alt uzay (F_q^n içinde).

        Dönüş Değeri
        ------------
        bool
            Aday uzayın tüm baz vektör çiftleri (u, w) için her P_k
            matrisinde u^T * P_k * w = 0 koşulunu sağlaması durumunda
            True; aksi halde False.
        """
        basis = space.basis()

        for P_k in self.P_matrices:
            for u in basis:
                for w in basis:
                    if u * P_k * w != 0:
                        return False

        return True

    def _is_vector_compatible_with_oil_candidate(self, vec, current_space):
        """
        Bir vektörün mevcut oil adayı uzaya eklenip eklenemeyeceğini
        açık anahtar matrisleriyle kontrol eder.

        Kontrol:
        Yeni oluşacak aday uzayın tüm baz vektörleri u,w için
        u^T P_k w = 0 koşulu sağlanmalıdır.

        Bu yardımcı metot, run_attack() içindeki adım adım "aday
        vektörleri deneyerek Oil uzayını inşa etme" sürecinin çekirdek
        mantığını oluşturur: bir vektör, yalnızca (a) mevcut uzaya
        yeni bir boyut katıyorsa, (b) hedef boyutu (o) aşmıyorsa ve
        (c) genişletilmiş uzay hâlâ _is_public_oil_candidate testini
        geçiyorsa kabul edilir.

        Parametreler
        ------------
        vec : vector
            Mevcut aday uzaya eklenmesi denenen F_q^n vektörü.
        current_space : VectorSpace
            Şu ana kadar doğrulanmış Oil uzayı adayı.

        Dönüş Değeri
        ------------
        bool
            vec, current_space'e eklenmeye uygunsa (yukarıdaki üç
            koşulu da sağlıyorsa) True; aksi halde False.
        """
        ambient = VectorSpace(self.F, self.n)

        # Sıfır vektörü eklenmez.
        if vec == ambient.zero():
            return False

        # Mevcut uzaya vec vektörünü ekleyerek deneme uzayı oluşturuyoruz.
        trial_space = current_space + ambient.span([vec])

        # Vektör zaten mevcut uzayın içindeyse yeni boyut eklemez.
        if trial_space.dimension() == current_space.dimension():
            return False

        # Oil uzayının hedef boyutu o'dur; daha büyük uzaylara izin vermiyoruz.
        if trial_space.dimension() > self.o:
            return False

        # Yeni uzay açık anahtar matrisleriyle oil adayı testini geçmeli.
        return self._is_public_oil_candidate(trial_space)

    def _extend_to_full_basis(self, subspace_basis):
        """
        Verilen bağımsız vektörleri standart baz vektörleriyle tamamlayarak
        F_q^n için tam bir baz üretir.

        Bu yardımcı metot, denk anahtar inşası (ADIM 4) sırasında
        bulunan Oil uzayının bazını, geri kalan (Vinegar'a karşılık
        gelecek) tamamlayıcı vektörlerle genişletmek için kullanılır.
        Standart baz vektörleri sırayla denenir; rank artırıyorsa
        (yani mevcut kümeden doğrusal bağımsızsa) eklenir.

        Parametreler
        ------------
        subspace_basis : list of vector
            Tamamlanacak, doğrusal bağımsız başlangıç vektörleri
            (tipik olarak bulunan Oil uzayının bazı).

        Dönüş Değeri
        ------------
        list of vector
            Uzunluğu n olan, F_q^n için tam bir baz oluşturan vektör
            listesi (ilk kısmı subspace_basis'i içerir).
        """
        full_basis = list(subspace_basis)
        ambient = VectorSpace(self.F, self.n)

        for e in ambient.basis():
            candidate = full_basis + [e]
            if matrix(self.F, candidate).rank() > len(full_basis):
                full_basis.append(e)

            if len(full_basis) == self.n:
                break

        return full_basis

    def _check_keys_generated(self):
        """
        Sistem inşa edilmeden saldırının çalıştırılmasını engeller.

        İstisnalar (Raises)
        --------------------
        RuntimeError
            T_mat henüz None ise veya P_matrices/P_polys listeleri
            boşsa (yani generate_keys() hiç çağrılmamışsa) fırlatılır.
        """
        if self.T_mat is None or len(self.P_matrices) == 0 or len(self.P_polys) == 0:
            raise RuntimeError("Önce generate_keys() çağrılmalıdır.")

    def run_attack(self):
        """
        Açık anahtar (P_matrices, P_polys) üzerinden Kipnis-Shamir
        saldırısını uçtan uca yürütür: gizli Oil uzayının yeniden inşası,
        denk anahtar çiftinin (T', F') oluşturulması ve sahte imza
        üretimi (forgery).

        Algoritmanın Adımları
        -----------------------
        1. Doğrusal Kombinasyonlarla W1, W2 Oluşturma:
           Açık anahtar matrislerinin (P_matrices) rastgele katsayılı
           doğrusal kombinasyonlarından iki matris (W1, W2) oluşturulur.
           W1'in TERSİNİR olması zorunludur (aksi halde W1^{-1}
           tanımsızdır); bu nedenle W1 tersinir olana kadar denemeler
           tekrarlanır (en fazla 50 deneme).

        2. Matris Demeti (Pencil) Analizi:
           W = W1^{-1} * W2 hesaplanır. Kipnis-Shamir'in temel teoremine
           göre, balanced OV'nin merkezi dönüşümündeki Oil x Oil sıfır
           bloğu yapısı nedeniyle, W matrisinin gizli Oil uzayını
           DEĞİŞMEZ ALT UZAY olarak barındırdığı ve bu uzayın W'nin
           karakteristik polinomunun çarpanlarına yansıdığı gösterilir.

        3. Karakteristik Polinomun Kareköküyle Aday Uzay Bulma:
           W'nin karakteristik polinomu C(lambda) hesaplanır ve
           çarpanlarına ayrılır. q tek balanced OV durumunda beklenen
           yapı C(lambda) = C1(lambda)^2 biçiminde TAM KAREDİR (her
           çarpanın üssü çifttir). Bu koşul sağlanmazsa (is_square =
           False), seçilen W1, W2 kombinasyonu uygun değildir ve
           fonksiyon erken sonlanır. Koşul sağlanırsa,
           C1(lambda) = sqrt(C(lambda)) polinomu her çarpanın üssü
           yarıya indirilerek yeniden inşa edilir ve C1(W) matrisi
           hesaplanır (Cayley-Hamilton temelli matris polinomu
           değerlendirmesi). C1(W)'nin sağ çekirdeği (right kernel),
           gizli Oil uzayına dair ADAY bir alt uzay verir.

        4. Doğrulama ve Birleştirme:
           Aday uzayın baz vektörleri, _is_vector_compatible_with_oil_candidate
           yardımcı metoduyla tek tek denenerek, yalnızca açık anahtar
           bilgisiyle (u^T P_k w = 0 testi) doğrulanabilen vektörler bir
           araya getirilir ve reconstructed_oil_space adım adım inşa
           edilir. Hedef boyuta (o) ulaşılamazsa saldırı bu W1, W2
           seçimiyle başarısız kabul edilir.

        5. Simülasyon Doğrulaması (Yalnızca Öğretici Amaçlı):
           Bulunan uzay, gizli Secret_Oil_Space ile karşılaştırılır. Bu
           karşılaştırma GERÇEK bir saldırgan tarafından YAPILAMAZ;
           yalnızca bu öğretici simülasyonun doğruluğunu teyit etmek
           içindir. Ayrıca bulunan bazın her vektörünün, açık anahtar
           polinomlarının TAMAMINI sıfırladığı (P_k(v) = 0, tüm k için)
           ekstra bir kontrolle doğrulanır.

        6. Denk Anahtar (Equivalent Key) İnşası:
           Bulunan Oil uzayının bazı, _extend_to_full_basis ile
           F_q^n için tam bir baza (Vinegar tamamlayıcı kısım + Oil
           bazı) genişletilir ve bu baz sütun olarak dizilerek saldırganın
           kendi T' matrisi elde edilir. Yeni merkezi dönüşüm matrisleri,
           F'_k = T'^T * P_k * T' formülüyle hesaplanır; bu matrislerin
           sağ alt o x o (Oil x Oil) bloğunun SIFIR olması beklenir ve
           bu, denk anahtarın doğruluğunun somut bir kanıtıdır.

        7. Sahte İmza (Forgery):
           Denk anahtar (T', F') kullanılarak, rastgele bir mesaj için
           standart UOV imzalama prosedürü (Vinegar rastgele seçilir,
           Oil değişkenlerine göre doğrusal sistem kurulup çözülür)
           uygulanır. Elde edilen imza, ORİJİNAL açık anahtar
           polinomlarında (self.P_polys) değerlendirilerek doğrulanır;
           eşleşme sağlanırsa saldırının sistemi tamamen kırdığı
           sonucuna varılır.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated ile denetlenir).

        Dönüş Değeri
        ------------
        None
            Bu metot, tüm sonuçları (aday uzaylar, denk anahtar, sahte
            imza ve doğrulama sonuçları) ayrıntılı biçimde ekrana
            yazdırır; bir değer döndürmez. Saldırı herhangi bir adımda
            başarısız olursa (tersinir W1 bulunamaması, karakteristik
            polinomun tam kare olmaması, Oil uzayının hedef boyuta
            ulaşamaması veya bulunan uzayın gizli uzayla örtüşmemesi
            gibi) erken bir `return` ile sonlanır.

        İstisnalar (Raises)
        --------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse fırlatılır.
        """
        self._check_keys_generated()
        print("\n" + "*"*30 + " ADIM 2: SALDIRI (CRYPTANALYSIS) " + "*"*30)

        # 1. AÇIK ANAHTAR MATRİSLERİNDEN LINEER KOMBİNASYONLAR
        # W1 ve W2, açık anahtar matrislerinin rastgele katsayılı doğrusal
        # kombinasyonlarıdır. W1'in tersinir olması, bir sonraki adımda
        # W = W1^{-1} * W2 matris demeti  tanımlanabilmesi
        # için zorunludur.
        print(f"\n[2.1] Açık Anahtar Matrislerinden Rastgele Lineer Kombinasyonlar Oluşturuluyor...")

        max_combination_attempts = 50

        for attempt in range(1, max_combination_attempts + 1):
            W1 = matrix(self.F, self.n, self.n)
            W2 = matrix(self.F, self.n, self.n)

            c1 = [self.F.random_element() for _ in range(self.o)]
            c2 = [self.F.random_element() for _ in range(self.o)]

            print(f"\n   Lineer kombinasyon denemesi {attempt}")
            print(f"   Rastgele Katsayılar (alfa): {c1}")
            print(f"   Rastgele Katsayılar (beta): {c2}")

            for k in range(self.o):
                W1 += c1[k] * self.P_matrices[k]
                W2 += c2[k] * self.P_matrices[k]
                print(f"      Adım {k}: W1 += {c1[k]} * P_{k}, W2 += {c2[k]} * P_{k}")

            if W1.is_invertible():
                print("   [ONAY] W1 tersinir bulundu.")
                break
        else:
            print("   [HATA] Tersinir bir W1 lineer kombinasyonu bulunamadı. Saldırı durduruluyor.")
            return

        print(f"\n   -> Oluşan W1 Matrisi:\n{W1}")
        print(f"   -> Oluşan W2 Matrisi:\n{W2}")

        # 2. W MATRİSİ
        # W = W1^{-1} * W2, Kipnis-Shamir'in "matris demeti" tekniğinin
        # merkezindeki nesnedir; gizli Oil uzayını değişmez alt uzay
        # olarak barındırdığı gösterilebilir.
        print(f"\n[2.2] W = W1^(-1) * W2 Hesaplanıyor...")
        W1_inv = W1.inverse()
        W = W1_inv * W2

        print(f"   -> W1^(-1) Matrisi:\n{W1_inv}")
        print(f"   -> W Matrisi:\n{W}")

        # 3. KARAKTERİSTİK POLİNOM
        # W'nin özyapısı (karakteristik polinomu), gizli Oil uzayına
        # dair bilgiyi taşır. q tek balanced OV durumunda bu polinomun
        # TAM KARE olması beklenir; bu, merkezi dönüşümün blok
        # simetrisinden kaynaklanan cebirsel bir sonuçtur.
        print(f"\n[2.3] Değişmez Alt Uzay Analizi")
        char_poly = W.characteristic_polynomial()
        print(f"   Karakteristik Polinom: {char_poly}")

        factors = char_poly.factor()
        print(f"   Çarpanlar: {factors}")

        # q tek balanced OV durumunda beklenen yapı:
        # C(lambda) = C1(lambda)^2.
        half_factors = []
        is_square = True

        for factor, exponent in factors:
            if exponent % 2 != 0:
                is_square = False
                break
            half_factors.append((factor, exponent // 2))

        if not is_square:
            print("   [UYARI] Karakteristik polinom tam kare değil. Yeni matris kombinasyonu denenmelidir.")
            return None

        # C1(lambda) = sqrt(C(lambda)) polinomunu oluşturuyoruz.
        poly_ring = char_poly.parent()
        C1 = poly_ring(1)

        for factor, half_exp in half_factors:
            C1 *= factor ** half_exp

        print(f"   C1(lambda) = sqrt(C(lambda)): {C1}")

        # Tezdeki algoritmaya uygun olarak C1(W) hesaplanır.
        # C1(W)'nin sağ çekirdeği, gizli Oil uzayının bir temsilini
        # (aday alt uzayı) verir; bu, saldırının cebirsel çekirdek
        # adımıdır.
        C1_W = C1(W)
        print(f"   C1(W) Matrisi:\n{C1_W}")

        candidate_space = C1_W.right_kernel()

        print(f"   Ker(C1(W)) Boyutu: {candidate_space.dimension()}")
        print(f"   Ker(C1(W)) Bazı:\n{candidate_space.basis_matrix()}")

        candidates = [candidate_space]

        # 5. DOĞRULAMA ve BİRLEŞTİRME
        # Aday uzayın vektörleri, yalnızca açık anahtar bilgisiyle
        # (u^T P_k w = 0 testi) tek tek doğrulanarak gerçek Oil uzayı
        # adım adım (o boyutuna ulaşana kadar) yeniden inşa edilir.
        print("\n" + "*"*30 + " ADIM 3: SONUÇ KONTROLÜ VE BİRLEŞTİRME " + "*"*30)

        ambient = VectorSpace(self.F, self.n)
        reconstructed_oil_space = ambient.subspace([])

        for idx, space in enumerate(candidates):
            print(f"\n[Aday {idx+1}]:")
            print(f"   Aday uzay boyutu: {space.dimension()}")

            added_from_this_candidate = 0

            # Aday uzayın baz vektörlerini tek tek deniyoruz.
            for vec in space.basis():
                if reconstructed_oil_space.dimension() == self.o:
                    break

                if self._is_vector_compatible_with_oil_candidate(vec, reconstructed_oil_space):
                    reconstructed_oil_space = reconstructed_oil_space + ambient.span([vec])
                    added_from_this_candidate += 1
                    print(f"   -> [EKLENDİ] {vec}")
                else:
                    print(f"   -> [RED] {vec}")

            if added_from_this_candidate == 0:
                print("   -> Bu adaydan yeni uyumlu vektör eklenemedi.")

        print(f"\n[3.1] Yeniden İnşa Edilen Oil Uzayı (Toplam):")
        print(f"      Boyut: {reconstructed_oil_space.dimension()} (Hedef: {self.o})")
        print(f"      Baz:\n{reconstructed_oil_space.basis_matrix()}")

        if reconstructed_oil_space.dimension() < self.o:
            print("   [UYARI] Bu W1, W2 seçimi oil uzayının tamamını vermedi.")
            print("        Yeni rastgele lineer kombinasyonlar denenmelidir.")
            return

        # Bu kontrol gerçek saldırgan tarafından yapılamaz.
        # Yalnızca simülasyon doğrulaması içindir.
        if reconstructed_oil_space != self.Secret_Oil_Space:
            print("\n[UYARI] Boyut doğru olsa da bulunan uzay gizli oil uzayıyla aynı değil.")
            print("        Bu kontrol yalnızca simülasyon doğrulaması içindir.")
            return
        # ... (run_attack fonksiyonunun sonuna ekle) ...

        # EKSTRA KONTROL: bulunan Oil uzayının her baz vektörünün, açık
        # anahtar polinomlarının TAMAMINI (tüm P_k için) sıfırladığı
        # doğrudan polinom değerlendirmesiyle teyit edilir. Bu, Oil
        # uzayının tanımlayıcı özelliğinin (u^T P_k u = 0) somut bir
        # gösterimidir.
        print("\n" + "*"*30 + " EKSTRA KONTROL: BAZ VEKTÖRLERİ SIFIRLANIYOR MU? " + "*"*30)

        # Bulunan Oil uzayının baz vektörleri
        found_basis = reconstructed_oil_space.basis()

        for vec_idx, vec in enumerate(found_basis):
            print(f"\n>>> Baz Vektörü {vec_idx + 1}: {vec}")

            is_zero_vector = True
            results = []

            for k, poly in enumerate(self.P_polys):
                # Vektörü polinoma sok (değişkenleri yerine koy)
                sub_dict = {self.vars[i]: vec[i] for i in range(self.n)}
                val = poly.subs(sub_dict)
                results.append(val)

                if val != 0:
                    is_zero_vector = False

            print(f"    P(v) Sonuçları: {results}")

            if is_zero_vector:
                print("    [ONAY] Bu vektör tüm polinomları SIFIRLIYOR.")
            else:
                print("    [HATA] Bu vektör polinomları sıfırlamıyor!")
        # 6. DENK ANAHTAR OLUŞTURMA (EQUIVALENT KEY)
        # Bulunan Oil uzayı, orijinal gizli anahtarla AYNI olmasa da
        # ONUNLA MATEMATİKSEL OLARAK EŞDEĞER yeni bir (T', F') anahtar
        # çifti kurmak için yeterlidir. Bu, MQ tabanlı imza şemalarına
        # yönelik yapısal saldırıların ortak sonlanma biçimidir: gizli
        # anahtarın KENDİSİ değil, ona denk bir kopyası elde edilir.
        print("\n" + "*"*30 + " ADIM 4: DENK ANAHTAR (T', F') İNŞASI " + "*"*30)
        print("Saldırgan, bulduğu Oil uzayını kullanarak kendi T' matrisini kurar.")

        # A) Yeni Baz T'
        oil_basis = list(reconstructed_oil_space.basis())

        # Önce oil bazını tam baza tamamlıyoruz.
        full_basis = self._extend_to_full_basis(oil_basis)

        # Tam bazın ilk n-o vektörünü vinegar tamamlayıcı baz olarak alıyoruz.
        # Son o vektör oil bazı olacak şekilde yeniden düzenleme yapıyoruz.
        extra_basis = [b for b in full_basis if b not in oil_basis]
        vinegar_basis = extra_basis[:self.v]

        T_prime_cols = list(vinegar_basis) + list(oil_basis)
        T_prime = matrix(self.F, T_prime_cols).transpose()

        print(f"\n[4.1] Saldırganın Oluşturduğu T' Matrisi:")
        print(T_prime)

        # B) Yeni Merkezi Dönüşüm F'
        # T' matrisinin sütunları, tanım gereği (Vinegar tamamlayıcı |
        # Oil bazı) sırasıyla dizildiğinden, F'_k = T'^T * P_k * T'
        # dönüşümü, açık anahtarı SALDIRGANIN kendi (varsayımsal)
        # Vinegar/Oil koordinat sistemine geri taşır. Bu matrislerin
        # sağ alt o x o bloğunun sıfır çıkması, T' ve bulunan Oil
        # uzayının DOĞRU olduğunun somut kanıtıdır.
        print(f"\n[4.2] Yeni Merkezi Matrislerin (F') Hesaplanması:")
        print(f"      DÜZELTME: F' = T'.transpose * P * T' formülü kullanılıyor.")
        print(f"      Beklenti: F' matrislerinin sağ alt {self.o}x{self.o} bloğu SIFIR olmalıdır.")

        F_prime_matrices = []
        for k in range(self.o):
            P_k = self.P_matrices[k]

            # *** DÜZELTİLEN FORMÜL ***
            # T' matrisinin sütunları zaten Oil vektörleridir.
            # Bu yüzden P'yi T' ile çarpmak, Oil vektörlerini P'ye sokmak demektir.
            F_prime = T_prime.transpose() * P_k * T_prime

            F_prime_matrices.append(F_prime)

            print(f"\n   >>> F'_{k} Matrisi:")
            print(F_prime)

            # Kontrol: Sağ alt blok (o x o) sıfır mı?
            oil_block = F_prime.submatrix(self.v, self.v, self.o, self.o)
            print(f"       Oil Blok:\n{oil_block}")

            if oil_block.is_zero():
                print("       [ONAY] Oil x Oil bloğu SIFIR. Denk Anahtar Doğru!")
            else:
                print("       [HATA] Blok sıfır değil!")

        # 7. SAHTE İMZA (FORGERY)
        # Denk anahtar (T', F') elde edildiğine göre, artık standart
        # UOV imzalama prosedürü (Vinegar sabitlenir, Oil değişkenlerine
        # göre doğrusal sistem kurulup çözülür) doğrudan uygulanabilir;
        # bu da saldırının nihai hedefi olan "gizli anahtarsız geçerli
        # imza üretme" yeteneğini kanıtlar.
        print("\n" + "*"*30 + " ADIM 5: SAHTE İMZA (FORGERY) " + "*"*30)

        msg = random_vector(self.F, self.o)
        print(f"   İmzalanacak Mesaj (y): {msg}")

        attempts = 0
        while attempts < 50:
            attempts += 1

            v_val = random_vector(self.F, self.v)

            coeffs = []
            consts = []

            for k in range(self.o):
                F_mat = F_prime_matrices[k]

                # Denklem Kurulumu (UOV Prosedürü)
                # Denklem: v^T * F_vv * v + v^T * (F_vo + F_ov^T) * o = y_k
                # Değişken: o (Oil vektörü)

                # F_vv (Sol Üst)
                F_vv = F_mat.submatrix(0, 0, self.v, self.v)
                # F_vo (Sağ Üst) + F_ov (Sol Alt) Simetrisi
                # F_cross = F_vo + F_ov.transpose()
                # (Genel durumda F_k simetrik olmayabilir ama x^T F x formunda simetrik kısmı önemlidir)
                # En garantisi: F_sym = F + F^T üzerinden gitmek
                F_sym = F_mat + F_mat.transpose()
                # Oil katsayıları bloğu (V x O)
                F_vo_sym = F_sym.submatrix(0, self.v, self.v, self.o)

                # Satır: v * F_vo_sym
                row = v_val * F_vo_sym
                coeffs.append(row)

                # Sabit terim: v * F_vv * v^T
                const_val = v_val * F_vv * v_val

                # Sağ taraf
                consts.append(msg[k] - const_val)

            A = matrix(self.F, coeffs)
            b = vector(self.F, consts)
            # =================== EKLEME BAŞLANGICI ===================
            print(f"\n   --- [Deneme {attempts}] Oluşan Lineer Sistem (Ax = b) ---")
            # 1. Matris ve Vektörün Ham Hali
            print(f"   A Matrisi (Katsayılar):\n{A}")
            print(f"   b Vektörü (Sabitler): {b}")

            # 2. Okunabilir Denklem Formatı
            print("   Açık Denklemler:")
            # Oil değişkenlerini isimlendirelim (o0, o1, ...)
            oil_var_names = [f"oil_{j}" for j in range(self.o)]

            for i in range(A.nrows()):
                equation_parts = []
                for j in range(A.ncols()):
                    coeff = A[i, j]
                    # Sadece 0 olmayan katsayıları yazdıralım ki kalabalık olmasın
                    if coeff != 0:
                        equation_parts.append(f"({coeff})*{oil_var_names[j]}")

                # Eğer tüm katsayılar 0 ise sol taraf 0'dır
                lhs = " + ".join(equation_parts) if equation_parts else "0"
                print(f"      Denklem {i+1}: {lhs} = {b[i]}")
            print("   -------------------------------------------------------")
            # =================== EKLEME BİTİŞİ =======================
            try:
                oil_val = A.solve_right(b)
                print(f"\n   [DENEME {attempts}] Lineer Sistem Çözüldü!")
                print(f"      Vinegar (v'): {v_val}")
                print(f"      Oil (o'): {oil_val}")

                # Birleştir: z' = (v', o')
                z_prime = vector(self.F, list(v_val) + list(oil_val))

                # Gerçek İmza: x = T' * z'
                signature = T_prime * z_prime
                print(f"      Sahte İmza (x): {signature}")

                # DOĞRULAMA
                # Üretilen sahte imza, saldırganın kendi denk anahtarıyla
                # DEĞİL, ORİJİNAL açık anahtar polinomlarıyla (self.P_polys)
                # denetlenir; bu, gerçek doğrulayıcının bakış açısını
                # simüle eder.
                print("\n   [DOĞRULAMA] Orijinal Açık Anahtar ile Kontrol:")
                check_res = []
                for poly in self.P_polys:
                    sub_dict = {self.vars[i]: signature[i] for i in range(self.n)}
                    val = poly.subs(sub_dict)
                    check_res.append(val)

                check_vec = vector(self.F, check_res)
                print(f"      P(imza) = {check_vec}")
                print(f"      Hedef   = {msg}")

                if check_vec == msg:
                    print("      >>> BAŞARILI! Sistem tamamen kırıldı.")
                    return

            except ValueError:
                pass



In [9]:
# ============================================================================
# SENARYO - SİSTEM KURULUMU
# ============================================================================
# Bu blok, KipnisShamirAttackDemo sınıfının tam bir yaşam döngüsünü
# (anahtar üretimi -> saldırı) örneklendiren bir DEMO/TEST senaryosudur.
print("\n\n" + "*"*60)
print(">>> KIPNIS-SHAMIR SİMÜLASYONU <<<")
print("*"*60)

# Bu öğretici örnekte q tek seçilmiştir.
# Karakteristik 2 durumunda simetrik matris temsili ve çapraz terimler ayrıca ele alınmalıdır.
my_q = 7
my_o = 3

ks = KipnisShamirAttackDemo(q=my_q, o=my_o)
ks.generate_keys()




************************************************************
>>> KIPNIS-SHAMIR SİMÜLASYONU <<<
************************************************************

KIPNIS-SHAMIR SALDIRISI: DÜZELTİLMİŞ TAM ANALİZ
Parametreler: GF(7), o=3, v=3 (Balanced), n=6

****************************** ADIM 1: SİSTEM İNŞASI (KEYGEN) ******************************

[1.1] Gizli T Matrisi (Değişken Karıştırıcı):
[2 5 1 3 1 0]
[5 4 2 4 3 1]
[3 0 3 1 0 5]
[4 3 3 3 2 3]
[6 5 3 0 0 0]
[1 2 6 4 3 2]

[1.2] Gizli Oil Uzayı O (Hedef):
   Baz Vektörleri:
[1 0 0 4 0 0]
[0 1 0 3 0 5]
[0 0 1 4 1 0]

[1.3] Gizli (F) ve Açık (P) Bileşenlerin İnşası

   >>> Bileşen k=0 <<<
   Gizli F_0 Matrisi (Sağ Alt Blok Sıfır):
[1 2 3 1 6 4]
[2 4 6 3 6 1]
[3 6 2 4 4 0]
[1 3 4 0 0 0]
[6 6 4 0 0 0]
[4 1 0 0 0 0]
   Açık P_0 Matrisi (Sıfırlar Kayboldu!):
[3 1 1 3 1 5]
[1 4 2 4 4 3]
[1 2 2 1 4 5]
[3 4 1 4 4 1]
[1 4 4 4 5 2]
[5 3 5 1 2 4]
   Açık Anahtar Polinomu Formülü: P_0(x) = x^T * P_0 * x
   Açık Anahtar Polinomu P_0(x) (Açık Hali):

In [11]:
# ============================================================================
# AYNI AÇIK ANAHTAR ÜZERİNDE SALDIRI DENEMESİ
# ============================================================================
ks.run_attack()



****************************** ADIM 2: SALDIRI (CRYPTANALYSIS) ******************************

[2.1] Açık Anahtar Matrislerinden Rastgele Lineer Kombinasyonlar Oluşturuluyor...

   Lineer kombinasyon denemesi 1
   Rastgele Katsayılar (alfa): [6, 5, 4]
   Rastgele Katsayılar (beta): [5, 2, 2]
      Adım 0: W1 += 6 * P_0, W2 += 5 * P_0
      Adım 1: W1 += 5 * P_1, W2 += 2 * P_1
      Adım 2: W1 += 4 * P_2, W2 += 2 * P_2
   [ONAY] W1 tersinir bulundu.

   -> Oluşan W1 Matrisi:
[6 1 0 0 3 5]
[1 4 4 0 3 4]
[0 4 4 6 0 5]
[0 0 6 4 0 4]
[3 3 0 0 3 5]
[5 4 5 4 5 6]
   -> Oluşan W2 Matrisi:
[0 1 6 5 5 0]
[1 1 4 0 0 4]
[6 4 2 4 2 1]
[5 0 4 1 0 0]
[5 0 2 0 2 6]
[0 4 1 0 6 5]

[2.2] W = W1^(-1) * W2 Hesaplanıyor...
   -> W1^(-1) Matrisi:
[6 5 3 1 4 5]
[5 4 1 5 3 4]
[3 1 4 4 1 4]
[1 5 4 5 2 5]
[4 3 1 2 5 0]
[5 4 4 5 0 3]
   -> W Matrisi:
[6 1 2 1 4 2]
[1 1 1 6 1 6]
[1 1 3 0 0 6]
[1 0 0 5 5 5]
[2 4 0 5 4 1]
[4 2 0 4 2 0]

[2.3] Değişmez Alt Uzay Analizi
   Karakteristik Polinom: x^6 + 2*x^5 + 4*x^4 